**_Copyright Notice_**

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Merging International Match and FIFA Ranking Data

In [1]:
import pandas as pd

## Load FIFA Ranking Data

This is the preprocessed data.

In [2]:
df_rank = pd.read_parquet("./intermediate/rank.parquet")

In [3]:
df_rank

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation
rank_date,,,,,,,
2018-08-16,145,Afghanistan,AFG,1068.00,1068.00,0,AFC
2018-08-17,145,Afghanistan,AFG,1068.00,1068.00,0,AFC
2018-08-18,145,Afghanistan,AFG,1068.00,1068.00,0,AFC
2018-08-19,145,Afghanistan,AFG,1068.00,1068.00,0,AFC
2018-08-20,145,Afghanistan,AFG,1068.00,1068.00,0,AFC
...,...,...,...,...,...,...,...
2022-10-02,123,Zimbabwe,ZIM,1138.56,1138.56,0,CAF
2022-10-03,123,Zimbabwe,ZIM,1138.56,1138.56,0,CAF
2022-10-04,123,Zimbabwe,ZIM,1138.56,1138.56,0,CAF


## Remove Unnecessary Columns

In [4]:
df_rank.drop(columns=["country_abrv", "previous_points", "confederation"], inplace=True)

In [5]:
df_rank

,rank,country_full,total_points,rank_change
rank_date,,,,
2018-08-16,145,Afghanistan,1068.00,0
2018-08-17,145,Afghanistan,1068.00,0
2018-08-18,145,Afghanistan,1068.00,0
2018-08-19,145,Afghanistan,1068.00,0
2018-08-20,145,Afghanistan,1068.00,0
...,...,...,...,...
2022-10-02,123,Zimbabwe,1138.56,0
2022-10-03,123,Zimbabwe,1138.56,0
2022-10-04,123,Zimbabwe,1138.56,0


## Load International Match Data

In [6]:
df_match = pd.read_csv('../data/results.csv')

## Convert to DateTime

In [7]:
df_match['date'] = pd.to_datetime(df_match['date'])

In [8]:
df_match

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49440,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49441,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49442,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49443,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


## Merge Match and Ranking

### Merge Home Team

In [9]:
df_merge_home = pd.merge(
    df_match,
    df_rank,
    left_on=["date", "home_team"],
    right_on=["rank_date", "country_full"],
    how="inner"
)

In [10]:
df_merge_home

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank,country_full,total_points,rank_change
0,2018-08-18,Andorra,United Arab Emirates,0.0,0.0,Friendly,Grödig,Austria,True,130,Andorra,1120.00,0
1,2018-08-18,Grenada,Jamaica,1.0,5.0,Friendly,St. George's,Grenada,False,168,Grenada,980.00,0
2,2018-08-18,Guatemala,Cuba,1.0,0.0,Friendly,Quetzaltenango,Guatemala,False,146,Guatemala,1064.00,0
3,2018-08-20,Barbados,Jamaica,2.0,2.0,Friendly,Bridgetown,Barbados,False,160,Barbados,1005.00,0
4,2018-08-25,Anguilla,Saint Martin,1.0,2.0,Friendly,The Valley,Anguilla,False,206,Anguilla,868.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3610,2022-09-27,Norway,Serbia,0.0,2.0,UEFA Nations League,Oslo,Norway,False,36,Norway,1488.57,0
3611,2022-09-27,Sweden,Slovenia,1.0,1.0,UEFA Nations League,Stockholm,Sweden,False,20,Sweden,1563.44,0
3612,2022-09-27,Kosovo,Cyprus,5.0,1.0,UEFA Nations League,Pristina,Kosovo,False,106,Kosovo,1183.90,0
3613,2022-09-27,Greece,Northern Ireland,3.0,1.0,UEFA Nations League,Athens,Greece,False,49,Greece,1441.45,1


### Merge Away Team

In [11]:
df_merge_home_away = pd.merge(
    df_merge_home,
    df_rank,
    left_on=["date", "away_team"],
    right_on=["rank_date", "country_full"],
    suffixes=["_home", "_away"],
    how="inner"
)

In [12]:
df_merge_home_away

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,country_full_home,total_points_home,rank_change_home,rank_away,country_full_away,total_points_away,rank_change_away
0,2018-08-18,Andorra,United Arab Emirates,0.0,0.0,Friendly,Grödig,Austria,True,130,Andorra,1120.00,0,77,United Arab Emirates,1312.00,0
1,2018-08-18,Grenada,Jamaica,1.0,5.0,Friendly,St. George's,Grenada,False,168,Grenada,980.00,0,54,Jamaica,1400.00,0
2,2018-08-18,Guatemala,Cuba,1.0,0.0,Friendly,Quetzaltenango,Guatemala,False,146,Guatemala,1064.00,0,181,Cuba,940.00,0
3,2018-08-20,Barbados,Jamaica,2.0,2.0,Friendly,Bridgetown,Barbados,False,160,Barbados,1005.00,0,54,Jamaica,1400.00,0
4,2018-08-26,Barbados,Cuba,0.0,0.0,Friendly,Bridgetown,Barbados,False,160,Barbados,1005.00,0,181,Cuba,940.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3402,2022-09-27,Norway,Serbia,0.0,2.0,UEFA Nations League,Oslo,Norway,False,36,Norway,1488.57,0,25,Serbia,1549.53,0
3403,2022-09-27,Sweden,Slovenia,1.0,1.0,UEFA Nations League,Stockholm,Sweden,False,20,Sweden,1563.44,0,65,Slovenia,1372.48,0
3404,2022-09-27,Kosovo,Cyprus,5.0,1.0,UEFA Nations League,Pristina,Kosovo,False,106,Kosovo,1183.90,0,108,Cyprus,1180.52,1
3405,2022-09-27,Greece,Northern Ireland,3.0,1.0,UEFA Nations League,Athens,Greece,False,49,Greece,1441.45,1,58,Northern Ireland,1399.10,0


## Drop Duplicate Info

In [13]:
df_merge_home_away.drop(columns=["country_full_home", "country_full_away"], inplace=True)

In [14]:
df_merge_home_away

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,total_points_home,rank_change_home,rank_away,total_points_away,rank_change_away
0,2018-08-18,Andorra,United Arab Emirates,0.0,0.0,Friendly,Grödig,Austria,True,130,1120.00,0,77,1312.00,0
1,2018-08-18,Grenada,Jamaica,1.0,5.0,Friendly,St. George's,Grenada,False,168,980.00,0,54,1400.00,0
2,2018-08-18,Guatemala,Cuba,1.0,0.0,Friendly,Quetzaltenango,Guatemala,False,146,1064.00,0,181,940.00,0
3,2018-08-20,Barbados,Jamaica,2.0,2.0,Friendly,Bridgetown,Barbados,False,160,1005.00,0,54,1400.00,0
4,2018-08-26,Barbados,Cuba,0.0,0.0,Friendly,Bridgetown,Barbados,False,160,1005.00,0,181,940.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3402,2022-09-27,Norway,Serbia,0.0,2.0,UEFA Nations League,Oslo,Norway,False,36,1488.57,0,25,1549.53,0
3403,2022-09-27,Sweden,Slovenia,1.0,1.0,UEFA Nations League,Stockholm,Sweden,False,20,1563.44,0,65,1372.48,0
3404,2022-09-27,Kosovo,Cyprus,5.0,1.0,UEFA Nations League,Pristina,Kosovo,False,106,1183.90,0,108,1180.52,1
3405,2022-09-27,Greece,Northern Ireland,3.0,1.0,UEFA Nations League,Athens,Greece,False,49,1441.45,1,58,1399.10,0


## Type Conversion

In [15]:
df_merge_home_away = df_merge_home_away.astype(
    {
        "home_score": int,
        "away_score": int
    }
)    

In [16]:
df_merge_home_away.loc[:, ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'rank_home', 'rank_away']]

,date,home_team,away_team,home_score,away_score,rank_home,rank_away
0,2018-08-18,Andorra,United Arab Emirates,0,0,130,77
1,2018-08-18,Grenada,Jamaica,1,5,168,54
2,2018-08-18,Guatemala,Cuba,1,0,146,181
3,2018-08-20,Barbados,Jamaica,2,2,160,54
4,2018-08-26,Barbados,Cuba,0,0,160,181
...,...,...,...,...,...,...,...
3402,2022-09-27,Norway,Serbia,0,2,36,25
3403,2022-09-27,Sweden,Slovenia,1,1,20,65
3404,2022-09-27,Kosovo,Cyprus,5,1,106,108
3405,2022-09-27,Greece,Northern Ireland,3,1,49,58


## Save

In [17]:
df_merge_home_away.to_parquet("./intermediate/match_rank.parquet")